# VLE Component Property Database

Interactive notebook for browsing, searching, adding, and editing component
thermodynamic properties stored in the SQLite database.

**Prerequisites:**
```bash
# Initialize and seed the database (run once)
PYTHONPATH=python/src python -m vle.cli.main init
PYTHONPATH=python/src python -m vle.cli.main seed --source chapter4
```

All values use canonical units: T in **K**, P in **kPa** (absolute), V in **cm³/mol**.

In [ ]:
import sys
sys.path.insert(0, '../python/src')

from vle.db import (
    get_connection, init_db, get_db_path,
    get_component, list_components, search_components, upsert_component,
    get_kij, set_kij, get_activity_params, set_activity_params,
    get_experimental_vle,
    ComponentRecord,
)
from pathlib import Path
import pandas as pd

# Verify database exists
db_path = get_db_path()
if not db_path.exists():
    print(f"Database not found at {db_path}. Initializing...")
    init_db()
    from vle.db.seed import seed_chapter4
    seed_chapter4()
    print("Database created and seeded with Chapter IV data.")
else:
    print(f"Database found at {db_path}")

## Browse All Components

In [ ]:
# Load all components into a DataFrame for easy viewing
comps = list_components()
df = pd.DataFrame([
    {
        'Name': c.name, 'Formula': c.formula, 'CAS': c.cas_number,
        'MW (g/mol)': c.mw, 'Tc (K)': c.tc, 'Pc (kPa)': c.pc,
        'w': c.w, 'Zc': c.zc, 'Vc (cm3/mol)': c.vc, 'Tb (K)': c.tb,
        'Source': c.source,
    }
    for c in comps
])
print(f"Total components: {len(df)}")
df

## Search Components

In [ ]:
# Search by name, formula, or CAS number
query = "methane"  # <-- Change this to search for different compounds

results = search_components(query)
for c in results:
    print(c.summary())

## View Component Details

In [ ]:
# Show all properties for a specific component
component_name = "methane"  # <-- Change this

comp = get_component(component_name)
if comp:
    props = {
        'Name': comp.name,
        'Formula': comp.formula,
        'CAS Number': comp.cas_number,
        'MW (g/mol)': comp.mw,
        '--- Critical Properties ---': '',
        'Tc (K)': comp.tc,
        'Pc (kPa absolute)': comp.pc,
        'Acentric factor w': comp.w,
        'Zc': comp.zc,
        'Vc (cm3/mol)': comp.vc,
        '--- Boiling Point ---': '',
        'Tb (K)': comp.tb,
        '--- Liquid Volume ---': '',
        'ZRA (Rackett)': comp.zra,
        'wSRK (Thomson)': comp.w_srk,
        '--- Other ---': '',
        'Dipole (Debye)': comp.dipole_moment,
        'Delta ((cal/cm3)^0.5)': comp.delta,
        'PRSV K1': comp.prsv_k1,
        '--- Metadata ---': '',
        'Source': comp.source,
        'Notes': comp.notes,
    }
    pd.DataFrame(list(props.items()), columns=['Property', 'Value'])
else:
    print(f"Component '{component_name}' not found.")

## Binary Interaction Parameters (kij)

In [ ]:
# Look up kij for a pair
comp1 = "carbon dioxide"
comp2 = "n-butane"
eos = "PR"

kij_rec = get_kij(comp1, comp2, eos, temperature=357.57)
if kij_rec:
    print(f"kij({comp1}/{comp2}, {eos}) = {kij_rec.kij}")
    print(f"Temperature: {kij_rec.temperature} K")
    print(f"Source: {kij_rec.source}")
else:
    print(f"No kij found for {comp1}/{comp2} with {eos} EOS.")
    print("Default kij = 0.0 will be used in calculations.")

## Activity Model Parameters

In [ ]:
# Look up activity model parameters
comp1 = "methanol"
comp2 = "water"
model = "van_laar"

params = get_activity_params(comp1, comp2, model, temperature=298.0)
if params:
    print(f"{model} parameters for {comp1}(1)/{comp2}(2):")
    print(f"  A12 = {params.a12}")
    print(f"  A21 = {params.a21}")
    print(f"  T = {params.temperature} K")
    print(f"  Source: {params.source}")
else:
    print(f"No {model} parameters found for {comp1}/{comp2}.")

## Experimental VLE Data

In [ ]:
# View experimental VLE data for a system
system = "CO2/n-butane"

data = get_experimental_vle(system)
if data:
    df_exp = pd.DataFrame([
        {'P (kPa)': d.pressure, 'x1': d.x1, 'y1': d.y1, 'T (K)': d.temperature}
        for d in data
    ])
    print(f"Experimental VLE data for {system} ({len(data)} points):")
    display(df_exp)
else:
    print(f"No experimental data found for {system}.")

## Add / Edit a Component

Use the cell below to add a new component or update an existing one.
All values must be in canonical units.

In [ ]:
# Uncomment and modify to add a new component:

# new_comp = ComponentRecord(
#     name="ethylene",
#     formula="C2H4",
#     cas_number="74-85-1",
#     mw=28.054,
#     tc=282.35,        # K
#     pc=5041.8,        # kPa (absolute)
#     w=0.0862,         # acentric factor
#     zc=0.281,
#     vc=131.1,         # cm3/mol
#     tb=169.42,        # K
#     source="manual",
#     notes="Added via notebook",
# )
# upsert_component(new_comp)
# print(f"Component '{new_comp.name}' saved.")

## Set Binary Parameters

Add or update kij or activity model parameters.

In [ ]:
# Uncomment to set a kij value:

# set_kij("methane", "ethane", eos_model="PR", kij=0.0, source="default")
# print("kij saved.")

# Uncomment to set activity model parameters:

# set_activity_params(
#     "methanol", "water", model="wilson",
#     a12=0.4321, a21=0.1234, temperature=298.0,
#     source="literature"
# )
# print("Activity parameters saved.")

## Seed from `thermo` Library (Optional)

If you have the `thermo` library installed (`pip install thermo`),
you can seed additional compounds on demand.

In [ ]:
# Uncomment to seed specific compounds from thermo:

# from vle.db.seed import seed_from_thermo
# count = seed_from_thermo(["acetaldehyde", "formaldehyde", "dimethyl ether"])
# print(f"Seeded {count} compounds from thermo.")